In [1]:

import os
import re
import pandas as pd
import numpy as np
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings("ignore")



In [2]:


splits = {
    'train': 'task_a/task_a_training_set_1.parquet', 
    'validation': 'task_a/task_a_validation_set.parquet', 
    'test': 'task_a/task_a_test_set_sample.parquet'
}

base_path = "hf://datasets/DaniilOr/SemEval-2026-Task13/"

# Load each split into Pandas
train_data = pd.read_parquet(base_path + splits["train"])
val_data = pd.read_parquet(base_path + splits["validation"])
test_data = pd.read_parquet(base_path + splits["test"])

'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/main/task_a/task_a_training_set_1.parquet
Retrying in 1s [Retry 1/5].


In [3]:
train_data.head(2)

,code,generator,label,language
0,"(a, b, c, d) = [int(x) for x in input().split(...",human,0,Python
1,valid version for the language; all others can...,Qwen/Qwen2.5-Coder-1.5B,1,Python


In [4]:
X_train_text, y_train = train_data["code"], train_data["label"]
X_test_text, y_test = test_data["code"], test_data["label"]
X_val_text, y_val = val_data["code"], val_data["label"]

In [5]:
# after hypertuning of the model
best_params = {
    'max_features': 30000, 
    'ngram_range': (1, 1), 
    'min_df': 0.0015430189491796615, 
    'C': 8.853443301355558, 
    'solver': 'liblinear', 
    'class_weight': 'balanced'
    }

In [ ]:
tfidf = TfidfVectorizer(
        strip_accents="unicode",
        analyzer="word",
        max_features=best_params["max_features"],
        ngram_range=best_params["ngram_range"],
        min_df=best_params["min_df"]
    )


X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)
# validation data
X_val_tfidf  = tfidf.transform(X_val_text)